# W5D6: Semantic cache with embeddings (Ollama + cosine similarity)

A semantic cache stores `(question embedding, answer)` pairs. When a new question arrives, it is embedded and compared with the cached questions by **cosine similarity**. If the best match is at or above a threshold, the cached answer is returned and **the LLM is never called**.

This notebook:

1. builds a `SemanticCache` class (in-memory, with an optional TTL),
2. embeds 20 Q&A pairs with `nomic-embed-text` via Ollama,
3. tests paraphrased questions (should hit) and unrelated questions (should miss) at the lesson's threshold of **0.92**,
4. sweeps the threshold to show the trade-off between hit rate and wrong hits,
5. estimates the money a cache saves.

The lesson suggests Redis and sentence-transformers. Here the same idea runs with the embedding model already used in the project, and the similarity search is plain numpy, which is all a cache this small needs. In production the vectors would live in Redis (with a TTL) or a vector store.

In [3]:
import time

import numpy as np
import ollama

EMBED_MODEL = "nomic-embed-text"
THRESHOLD = 0.92   # lesson's value: cosine similarity >= 0.92 counts as a hit


def embed(texts):
    """Embed a list of texts with Ollama; returns an array of unit-length vectors."""
    vecs = np.array(ollama.embed(model=EMBED_MODEL, input=texts)["embeddings"], dtype=float)
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)  # normalise so dot product = cosine


class SemanticCache:
    """Cache keyed by meaning: returns a stored answer for a similar-enough question."""

    def __init__(self, threshold=THRESHOLD, ttl_seconds=None):
        self.threshold = threshold
        self.ttl = ttl_seconds          # None = entries never expire
        self.vectors = []               # unit-length question embeddings
        self.entries = []               # dicts: question, answer, created
        self.hits = 0
        self.misses = 0

    def put(self, question, answer, vector=None):
        vector = embed([question])[0] if vector is None else vector
        self.vectors.append(vector)
        self.entries.append({"question": question, "answer": answer, "created": time.time()})

    def lookup(self, question, vector=None):
        """Return (entry or None, best similarity, index of best match) without changing stats."""
        if not self.entries:
            return None, 0.0, None
        vector = embed([question])[0] if vector is None else vector
        sims = np.array(self.vectors) @ vector          # cosine similarity to every entry
        best = int(np.argmax(sims))
        entry = self.entries[best]
        expired = self.ttl is not None and time.time() - entry["created"] > self.ttl
        if sims[best] >= self.threshold and not expired:
            return entry, float(sims[best]), best
        return None, float(sims[best]), best

    def get(self, question, vector=None):
        """Return the cached answer or None, and update hit/miss counters."""
        entry, sim, _ = self.lookup(question, vector)
        if entry:
            self.hits += 1
            return entry["answer"], sim
        self.misses += 1
        return None, sim

    @property
    def hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total else 0.0

ModuleNotFoundError: No module named 'ollama'

## 1. The 20 Q&A pairs

In [ ]:
QA_PAIRS = [
    ("What is Ollama?", "Ollama is a tool for downloading and running large language models locally through a simple CLI and REST API."),
    ("How do I pull a model in Ollama?", "Run `ollama pull <model>`, for example `ollama pull llama3.2:3b`."),
    ("What is an embedding?", "An embedding is a dense vector of numbers that represents the meaning of a piece of text, so similar meanings sit close together."),
    ("What is cosine similarity?", "Cosine similarity measures the angle between two vectors; 1 means the same direction, 0 means unrelated."),
    ("What is ChromaDB?", "ChromaDB is an open-source vector database that stores embeddings with metadata and supports similarity search."),
    ("What is RAG?", "Retrieval-augmented generation retrieves relevant document chunks and gives them to an LLM as context for its answer."),
    ("Why do we chunk documents?", "Documents are split into chunks so that retrieval returns small, focused passages that fit the model's context window."),
    ("What is chunk overlap?", "Chunk overlap repeats a little text between neighbouring chunks so sentences cut at a boundary are not lost."),
    ("What does top-k mean in retrieval?", "Top-k is the number of most similar chunks returned for a query."),
    ("What is a system prompt?", "A system prompt is an instruction given to the model before the conversation that sets its role and rules."),
    ("What is temperature in an LLM?", "Temperature controls randomness: low values give predictable answers, high values give more varied ones."),
    ("What is quantisation?", "Quantisation stores model weights with fewer bits, shrinking memory use and speeding up inference at a small cost in quality."),
    ("What is a token?", "A token is a chunk of text, roughly four characters in English, that is the unit LLMs read and are billed by."),
    ("Why use a local LLM instead of a cloud API?", "A local LLM keeps data private and has no per-token fee, but is usually slower and less capable than large hosted models."),
    ("What is a vector database?", "A vector database stores embeddings and finds the nearest ones to a query vector quickly."),
    ("What is semantic search?", "Semantic search finds results by meaning using embeddings, rather than by matching exact keywords."),
    ("What is prompt compression?", "Prompt compression removes low-information tokens from a prompt to cut cost and latency with little quality loss."),
    ("What is a context window?", "The context window is the maximum number of tokens a model can read in one request."),
    ("What is hallucination in LLMs?", "Hallucination is when a model states something fluent but false or unsupported by its context."),
    ("What is semantic caching?", "Semantic caching returns a stored answer when a new question is similar enough in meaning to an earlier one, skipping the LLM call."),
]
print(len(QA_PAIRS), "Q&A pairs")

cache = SemanticCache(threshold=THRESHOLD)
question_vectors = embed([q for q, _ in QA_PAIRS])      # one batched call for all 20
for (q, a), v in zip(QA_PAIRS, question_vectors):
    cache.put(q, a, vector=v)
print("cache size:", len(cache.entries))

20 Q&A pairs


cache size: 20


## 2. Test: paraphrases should hit, unrelated questions should miss

Each paraphrase is tagged with the index of the cached question it should match. A **correct hit** matches the right entry, a **wrong hit** returns a different entry's answer (the dangerous case), and a **miss** means the LLM would be called.

In [ ]:
PARAPHRASES = [   # (new question, index of the cached question it should match)
    ("Explain what Ollama is", 0),
    ("What command downloads a model in Ollama?", 1),
    ("Define an embedding", 2),
    ("Explain cosine similarity", 3),
    ("Tell me about ChromaDB", 4),
    ("What does retrieval-augmented generation mean?", 5),
    ("Why do we split documents into chunks?", 6),
    ("What is a token in an LLM?", 12),
    ("Why run models locally rather than using a cloud API?", 13),
    ("What is quantization?", 11),
]
UNRELATED = [
    "Who won the 2010 FIFA World Cup?",
    "What is the capital of France?",
    "How do I bake sourdough bread?",
    "What is the tallest mountain in the world?",
    "What is the best way to learn guitar?",
]

para_vecs = embed([q for q, _ in PARAPHRASES])
unrel_vecs = embed(UNRELATED)

print(f"threshold = {THRESHOLD}\n")
print(f"{'new question':<55}{'best sim':>9}  result")
for (q, expected), v in zip(PARAPHRASES, para_vecs):
    entry, sim, best = cache.lookup(q, vector=v)
    result = "MISS" if entry is None else ("HIT (correct)" if best == expected else "HIT (WRONG)")
    print(f"{q:<55}{sim:>9.3f}  {result}")
print()
for q, v in zip(UNRELATED, unrel_vecs):
    entry, sim, best = cache.lookup(q, vector=v)
    print(f"{q:<55}{sim:>9.3f}  {'MISS (correct)' if entry is None else 'HIT (WRONG)'}")

threshold = 0.92

new question                                            best sim  result
Explain what Ollama is                                     0.966  HIT (correct)
What command downloads a model in Ollama?                  0.829  MISS
Define an embedding                                        0.957  HIT (correct)
Explain cosine similarity                                  0.962  HIT (correct)
Tell me about ChromaDB                                     0.956  HIT (correct)
What does retrieval-augmented generation mean?             0.580  MISS
Why do we split documents into chunks?                     0.869  MISS
What is a token in an LLM?                                 0.830  MISS
Why run models locally rather than using a cloud API?      0.787  MISS
What is quantization?                                      0.885  MISS

Who won the 2010 FIFA World Cup?                           0.401  MISS (correct)
What is the capital of France?                             0.407  MISS (correct)


## 3. Threshold sweep

A high threshold is safe but misses paraphrases; a low one hits more but starts returning answers to the wrong question. The right value is the lowest one with **zero wrong hits** on your data. The sweep below uses the similarities already computed, so it needs no extra embedding calls.

In [ ]:
def sweep(thresholds):
    rows = []
    for t in thresholds:
        correct = wrong = missed = false_hits = 0
        for (q, expected), v in zip(PARAPHRASES, para_vecs):
            sims = np.array(cache.vectors) @ v
            best = int(np.argmax(sims))
            if sims[best] >= t:
                correct += best == expected
                wrong += best != expected
            else:
                missed += 1
        for v in unrel_vecs:
            false_hits += float(np.max(np.array(cache.vectors) @ v)) >= t
        rows.append((t, correct, wrong, missed, false_hits))
    return rows


print(f"{'threshold':>9}{'correct hits':>14}{'wrong hits':>12}{'misses':>8}{'unrelated hit':>15}")
for t, c, w, m, fh in sweep([0.80, 0.84, 0.88, 0.90, 0.92, 0.94, 0.96]):
    print(f"{t:>9.2f}{c:>10}/{len(PARAPHRASES)}{w:>12}{m:>8}{fh:>12}/{len(UNRELATED)}")

threshold  correct hits  wrong hits  misses  unrelated hit
     0.80         8/10           0       2           0/5
     0.84         6/10           0       4           0/5
     0.88         5/10           0       5           0/5
     0.90         4/10           0       6           0/5
     0.92         4/10           0       6           0/5
     0.94         4/10           0       6           0/5
     0.96         2/10           0       8           0/5


## 4. Live demo: cache in front of the LLM

`ask()` checks the cache first and only calls the local model on a miss, then stores the new answer. This is where the saving comes from.

In [ ]:
CHAT_MODEL = "llama3.2:3b"
llm_calls = 0


def ask(question):
    """Cache first; call the local LLM only on a miss and remember the result."""
    global llm_calls
    answer, sim = cache.get(question)
    if answer is not None:
        return answer, f"CACHE HIT (sim {sim:.3f})"
    reply = ollama.chat(model=CHAT_MODEL, messages=[{"role": "user", "content": question}],
                        options={"temperature": 0, "num_predict": 120})
    llm_calls += 1
    answer = reply["message"]["content"].strip()
    cache.put(question, answer)
    return answer, f"LLM CALL (best sim {sim:.3f})"


for q in ["Explain what Ollama is",              # paraphrase of a cached question
          "What is the boiling point of water?",  # not cached: goes to the LLM
          "What is the boiling point of water?"]: # same question again: now cached
    a, source = ask(q)
    print(f"{q}\n  [{source}] {a[:110]}\n")

print(f"hits={cache.hits} misses={cache.misses} hit rate={cache.hit_rate:.0%} LLM calls={llm_calls}")

Explain what Ollama is
  [CACHE HIT (sim 0.966)] Ollama is a tool for downloading and running large language models locally through a simple CLI and REST API.



What is the boiling point of water?
  [LLM CALL (best sim 0.547)] The boiling point of water is 212 degrees Fahrenheit (°F) or 100 degrees Celsius (°C) at standard atmospheric 

What is the boiling point of water?
  [CACHE HIT (sim 1.000)] The boiling point of water is 212 degrees Fahrenheit (°F) or 100 degrees Celsius (°C) at standard atmospheric 

hits=2 misses=1 hit rate=67% LLM calls=1


## 5. Estimated saving from the cache

In [ ]:
# Same assumptions as the audit spreadsheet (lesson prices, USD per 1M tokens)
USD_INR = 88.0                      # update to the current exchange rate
IN_TOKENS, OUT_TOKENS = 600, 150    # typical RAG request: edit with your measured values
PRICES = {"GPT-4o mini": (0.15, 0.60), "GPT-4o": (5.00, 15.00)}

hits_on_paraphrases = sum(1 for (q, _), v in zip(PARAPHRASES, para_vecs) if cache.lookup(q, vector=v)[0])
measured_hit_rate = hits_on_paraphrases / len(PARAPHRASES)
print(f"Hit rate on the paraphrase test set at {THRESHOLD}: {measured_hit_rate:.0%}")
print("(real traffic will have a different hit rate; use this as the input to the audit sheet)\n")

for name, (p_in, p_out) in PRICES.items():
    per_call = (IN_TOKENS * p_in + OUT_TOKENS * p_out) / 1_000_000 * USD_INR
    saved_per_1000 = per_call * 1000 * measured_hit_rate
    print(f"{name:<12} INR {per_call:.4f} per call | saved per 1,000 queries at this hit rate: INR {saved_per_1000:,.2f}")

Hit rate on the paraphrase test set at 0.92: 40%
(real traffic will have a different hit rate; use this as the input to the audit sheet)

GPT-4o mini  INR 0.0158 per call | saved per 1,000 queries at this hit rate: INR 6.34
GPT-4o       INR 0.4620 per call | saved per 1,000 queries at this hit rate: INR 184.80


## Observations

- Hit rate at 0.92 on paraphrases: 4/10 (40%). Four paraphrases hit (Ollama 0.966, embedding 0.957, cosine similarity 0.962, ChromaDB 0.956) and six missed. The misses scored between 0.58 and 0.89, for example "What does retrieval-augmented generation mean?" scored only 0.580 against the cached "What is RAG?", because the acronym and its expansion look different to the embedding model.
- Wrong hits at 0.92 (should be 0): 0. There were no wrong hits at any threshold in the sweep (0.80 to 0.96), so every hit returned the answer to the right question.
- Lowest threshold with zero wrong hits: 0.80, the lowest value tested. At 0.80 there were 8/10 correct hits, 0 wrong hits and 0/5 unrelated hits. Hit rate rose as the threshold fell: 8/10 at 0.80, 6/10 at 0.84, 5/10 at 0.88, 4/10 at 0.90 to 0.94 and 2/10 at 0.96.
- Did any unrelated question hit? No. All 5 unrelated questions missed, with similarities between 0.390 and 0.434, far below every threshold tested.
- Threshold I would use in production and why: 0.88 as a starting point (5/10 hits, 0 wrong hits), then lower it step by step towards 0.80 to 0.84 once it has been tested on more questions. A wrong cached answer is worse than a missed saving, because a miss only costs one LLM call. The unrelated test questions were all off-topic, so they do not show how the cache behaves on on-topic questions with a different meaning (for example "What is a token?" vs "What is a context window?"). That case needs testing before going lower. The threshold also depends on the nomic-embed-text embeddings, so it must be re-tuned if the model changes.
- Limits of this cache (stale answers, TTL, context-dependent questions): Cached answers can go stale when the underlying documents or facts change, so entries need a TTL (the `ttl_seconds` option) or a flush when documents are re-indexed. Questions that depend on context ("what about the second one?", "summarise it") can have the same wording but different meanings, so they should not be cached. Paraphrases that differ in a key detail (a different number, name or negation) can score as very similar and return the wrong answer, which is why the threshold must be tuned. Paraphrases can also miss entirely when the wording differs a lot (the RAG example scored 0.580). The cache is in memory only and uses a linear scan, so it is lost on restart and slows down with many entries, which is why production systems use Redis or a vector store.